importações

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.metrics import accuracy_score

from evaluation_utils import (
    calcular_metricas_completas,
    salvar_matriz_confusao,
    exportar_relatorios,
    plotar_painel_curvas
)

Download do dataset tratado e pre-processamento

In [ ]:
df = pd.read_excel('./Entrega_1/base_churn_5000_VF2_aluno_preprocessada.xlsx')

#Retirada de id e churn pelo viés 
#DATA_AQUISICAO_CLIENTE deve virar tempo de contrato e DATA_OCORRENCIA_CHURN é para análise de 30, 60, 90 dias.
X = df.drop(columns=['id_cliente', 'churn'])
y = df['churn']

#Dividindo os dados em treino e teste, com 30% dos dados para teste e estratificando pelo target churn
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# One-hot encoding para variáveis categóricas tipo_contrato e forma_pagamento
categorical_features = ['tipo_contrato', 'forma_pagamento']
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

# Criando DataFrames para os dados codificados
df_encoded_train = pd.DataFrame(X_train_encoded.toarray(), columns=encoder.get_feature_names_out(categorical_features), index=X_train.index)
df_encoded_test = pd.DataFrame(X_test_encoded.toarray(), columns=encoder.get_feature_names_out(categorical_features), index=X_test.index)

#Concatenando os dados codificados com os dados originais, removendo as colunas categóricas originais
X_train_final = pd.concat([
    X_train.drop(columns=categorical_features),
    df_encoded_train
], axis=1)

X_test_final = pd.concat([
    X_test.drop(columns=categorical_features),
    df_encoded_test
], axis=1)

# Seleciona as colunas originais que são numéricas e podem conter nulos
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

imputer = KNNImputer(n_neighbors=5, metric='nan_euclidean')

# Regra de Ouro: Aprende (fit) APENAS no Treino para não ter Data Leakage
X_train_final[numeric_features] = imputer.fit_transform(X_train_final[numeric_features])

# Aplica as regras aprendidas (transform) no Teste
X_test_final[numeric_features] = imputer.transform(X_test_final[numeric_features])

Árvore de decisão (Scikit-Learn)

In [ ]:
model = DecisionTreeClassifier(random_state=42, max_depth=5)
model.fit(X_train_final, y_train)

y_prob = model.predict_proba(X_test_final)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

res_base = calcular_metricas_completas(
    "1_Baseline_Desbalanceada_Limiar_0.50",
    y_test, y_pred, y_prob,
    hiperparametros={"max_depth": 5, "criterion": "gini", "balanceamento": "Nenhum"},
    limiar=0.50
)

Variação do hiperparâmetro max_depth

In [ ]:
historico_depths = []
for d in range(1, 16):
    clf_d = DecisionTreeClassifier(random_state=42, max_depth=d)
    clf_d.fit(X_train_final, y_train)
    acc_tr = float(accuracy_score(y_train, clf_d.predict(X_train_final)))
    y_pr_te = clf_d.predict(X_test_final)
    y_pb_te = clf_d.predict_proba(X_test_final)[:, 1]

    res_d = calcular_metricas_completas(
        f"Exp_MaxDepth_{d:02d}",
        y_test, y_pr_te, y_pb_te,
        hiperparametros={"max_depth": d, "criterion": "gini"}
    )
    res_d["metricas"]["acuracia_treino"] = round(acc_tr, 4)
    res_d["metricas"]["gap_overfitting"] = round(acc_tr - res_d["metricas"]["acuracia"], 4)
    historico_depths.append(res_d)


Variação do hiperparâmetro min_samples

In [ ]:
historico_depths = []
for d in range(1, 100):
    clf_d = DecisionTreeClassifier(random_state=42, min_samples=d)
    clf_d.fit(X_train_final, y_train)
    acc_tr = float(accuracy_score(y_train, clf_d.predict(X_train_final)))
    y_pr_te = clf_d.predict(X_test_final)
    y_pb_te = clf_d.predict_proba(X_test_final)[:, 1]

    res_d = calcular_metricas_completas(
        f"Exp_MaxDepth_{d:02d}",
        y_test, y_pr_te, y_pb_te,
        hiperparametros={"max_depth": d, "criterion": "gini"}
    )
    res_d["metricas"]["acuracia_treino"] = round(acc_tr, 4)
    res_d["metricas"]["gap_overfitting"] = round(acc_tr - res_d["metricas"]["acuracia"], 4)
    historico_depths.append(res_d)
